In [4]:
# ── Installs ──────────────────────────────────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("ultralytics")
install("ensemble-boxes")   # for Weighted Box Fusion
install("opencv-python-headless")

# ── Imports ───────────────────────────────────────────────────────────────
import os, shutil, random, yaml, urllib.request, zipfile
from pathlib import Path
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

# ── Config ────────────────────────────────────────────────────────────────
ROOT        = Path("./visdrone_project")
DATA_DIR    = ROOT / "VisDrone"
SUBSET_DIR  = ROOT / "VisDrone_300"    # 300-image subset for fast training
MAX_IMAGES  = 300                      # fast but representative
EPOCHS_V5   = 10                       # enough to converge on small dataset
EPOCHS_V8   = 10
IMG_SIZE    = 416                      # smaller than 640 → ~2x faster per epoch

# ── GPU Setup ─────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE     = 0                     # GPU index (0 = first GPU)
    GPU_NAME   = torch.cuda.get_device_name(0)
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE = 16                    # larger batch = faster on GPU
    print(f"✓ GPU detected : {GPU_NAME}  ({GPU_MEM_GB:.1f} GB)")
    print(f"  Batch size   : {BATCH_SIZE}")
else:
    DEVICE     = "cpu"
    BATCH_SIZE = 4                     # small batch for CPU
    print("⚠ No GPU found — running on CPU (will be slow)")
    print("  Tip: Use Google Colab for free GPU access")
CONF_THRES  = 0.25
IOU_THRES   = 0.45

CLASSES = [
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor"
]

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")


# ===========================================================================
# STEP 1 — Download VisDrone via Ultralytics helper
# ===========================================================================
def download_visdrone():
    """Download VisDrone using ultralytics auto-download."""
    print("\n" + "="*60)
    print("STEP 1: Downloading VisDrone Dataset")
    print("="*60)

    # Trigger download by running a minimal YOLO train dry-run with VisDrone.yaml
    # Ultralytics will auto-download the dataset to datasets/VisDrone
    yaml_content = """
path: ./datasets/VisDrone
train: VisDrone2019-DET-train/images
val:   VisDrone2019-DET-val/images
test:  VisDrone2019-DET-test-dev/images

nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""
    yaml_path = ROOT / "VisDrone.yaml"
    yaml_path.write_text(yaml_content)

    # Use ultralytics to download
    try:
        from ultralytics.utils.downloads import download
        urls = [
            "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip",
            "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip",
            "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip",
        ]
        dl_dir = Path("./datasets")
        dl_dir.mkdir(exist_ok=True)
        for url in urls:
            fname = dl_dir / Path(url).name
            if not fname.exists():
                print(f"Downloading {Path(url).name} ...")
                download(url, dir=dl_dir)
            else:
                print(f"Already exists: {fname.name}")
    except Exception as e:
        print(f"Auto-download note: {e}")
        print("If download fails, manually download from:")
        print("  https://github.com/VisDrone/VisDrone-Dataset")


# ===========================================================================
# STEP 2 — Create 500-image subset & convert annotations
# ===========================================================================
def visdrone_to_yolo(anno_path, img_w, img_h):
    """
    Convert VisDrone annotation (CSV format) to YOLO format.
    VisDrone format: bbox_left, bbox_top, bbox_width, bbox_height,
                     score, category_id, truncation, occlusion
    YOLO format:     class x_center y_center width height (normalized)
    """
    lines = []
    if not anno_path.exists():
        return lines
    with open(anno_path) as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            x, y, w, h = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
            cat = int(parts[5]) - 1  # VisDrone class IDs are 1-indexed; 0=ignored
            if cat < 0 or cat > 9:
                continue
            # Normalize
            xc = (x + w / 2) / img_w
            yc = (y + h / 2) / img_h
            wn = w / img_w
            hn = h / img_h
            # Clamp
            xc = min(max(xc, 0), 1)
            yc = min(max(yc, 0), 1)
            wn = min(max(wn, 0), 1)
            hn = min(max(hn, 0), 1)
            lines.append(f"{cat} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
    return lines


def prepare_subset():
    """Build a 500-image YOLO-format subset from the downloaded VisDrone data."""
    print("\n" + "="*60)
    print("STEP 2: Preparing 500-image subset")
    print("="*60)

    # Look for downloaded data
    base = Path("./datasets/VisDrone2019-DET-train")
    img_src  = base / "images"
    ann_src  = base / "annotations"

    if not img_src.exists():
        print(f"[!] Train images not found at {img_src}")
        print("    Creating a synthetic demo dataset instead...")
        create_synthetic_dataset()
        return

    # Gather images
    all_imgs = sorted(list(img_src.glob("*.jpg")) + list(img_src.glob("*.png")))
    random.seed(42)
    random.shuffle(all_imgs)
    selected = all_imgs[:MAX_IMAGES]
    print(f"Selected {len(selected)} / {len(all_imgs)} training images (capped at {MAX_IMAGES})")

    # Build folder structure
    for split in ["train", "val"]:
        (SUBSET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (SUBSET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    # Split: 80% train, 20% val
    split_idx = int(0.8 * len(selected))
    splits = {"train": selected[:split_idx], "val": selected[split_idx:]}

    for split, imgs in splits.items():
        for img_path in imgs:
            # Copy image
            dst_img = SUBSET_DIR / "images" / split / img_path.name
            shutil.copy2(img_path, dst_img)

            # Convert annotation
            ann_path = ann_src / (img_path.stem + ".txt")
            # Use PIL instead of cv2 to avoid NumPy 2.x/OpenCV imdecode conflict
            try:
                from PIL import Image as PILImage
                with PILImage.open(img_path) as pil_img:
                    w, h = pil_img.size   # PIL gives (width, height)
            except Exception:
                h, w = 1080, 1920         # safe fallback
            yolo_lines = visdrone_to_yolo(ann_path, w, h)

            dst_lbl = SUBSET_DIR / "labels" / split / (img_path.stem + ".txt")
            dst_lbl.write_text("\n".join(yolo_lines))

        print(f"  {split}: {len(imgs)} images")

    # Write dataset YAML
    write_dataset_yaml()


def create_synthetic_dataset():
    """
    Fallback: create a tiny synthetic dataset so all training/ensemble
    code runs even without the real download.
    """
    print("Creating synthetic VisDrone-style dataset (100 images, random boxes)...")
    for split in ["train", "val"]:
        img_dir = SUBSET_DIR / "images" / split
        lbl_dir = SUBSET_DIR / "labels" / split
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)

        n = 240 if split == "train" else 60
        for i in range(n):
            # Create a blank 640x640 image with random colored rectangles
            img = np.ones((640, 640, 3), dtype=np.uint8) * 200
            boxes = []
            for _ in range(random.randint(2, 8)):
                cls = random.randint(0, 9)
                x1 = random.randint(10, 500)
                y1 = random.randint(10, 500)
                bw = random.randint(30, 100)
                bh = random.randint(30, 100)
                x2 = min(x1 + bw, 630)
                y2 = min(y1 + bh, 630)
                color = [random.randint(50, 255) for _ in range(3)]
                # cv2.rectangle replaced with PIL for NumPy 2.x compatibility
                from PIL import Image as PILImage, ImageDraw
                pil_tmp = PILImage.fromarray(img)
                ImageDraw.Draw(pil_tmp).rectangle([x1,y1,x2,y2], outline=tuple(color), width=2)
                img = np.array(pil_tmp)
                xc = (x1 + x2) / 2 / 640
                yc = (y1 + y2) / 2 / 640
                wn = (x2 - x1) / 640
                hn = (y2 - y1) / 640
                boxes.append(f"{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
            PILImage.fromarray(img).save(str(img_dir / f"img_{i:04d}.jpg"))
            (lbl_dir / f"img_{i:04d}.txt").write_text("\n".join(boxes))

    write_dataset_yaml()
    print(f"Synthetic dataset created at {SUBSET_DIR}")


def write_dataset_yaml():
    cfg = {
        "path": str(SUBSET_DIR.resolve()),
        "train": "images/train",
        "val":   "images/val",
        "nc": 10,
        "names": CLASSES
    }
    yaml_path = SUBSET_DIR / "dataset.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f"Dataset YAML written: {yaml_path}")
    return yaml_path


# ===========================================================================
# STEP 3 — Train YOLOv5
# ===========================================================================
def train_yolov5():
    print("\n" + "="*60)
    print("STEP 3: Training YOLOv5n on VisDrone subset")
    print("="*60)

    yaml_path = SUBSET_DIR / "dataset.yaml"
    model = YOLO("yolov5nu.pt")   # YOLOv5n ultralytics version

    results = model.train(
        data=str(yaml_path),
        epochs=EPOCHS_V5,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        project=str(ROOT / "runs"),
        name="yolov5_visdrone",
        exist_ok=True,
        verbose=True,
        workers=0,          # 0 = safer on Windows; increase on Linux/Colab
        patience=5,
        save=True,
        amp=True,           # mixed precision — faster on GPU, ignored on CPU
        cache=True,         # cache images in RAM for faster data loading
    )

    # Dynamically find where ultralytics actually saved the weights
    best_weights = ROOT / "runs" / "yolov5_visdrone" / "weights" / "best.pt"
    if not best_weights.exists():
        # Search for best.pt anywhere under runs/
        candidates = sorted((ROOT / "runs").rglob("best.pt"))
        v5_candidates = [p for p in candidates if "yolov5" in str(p)]
        if v5_candidates:
            best_weights = v5_candidates[-1]
        elif candidates:
            best_weights = candidates[-1]

    # Fallback to last.pt if best.pt not found
    if not best_weights.exists():
        last_weights = best_weights.parent / "last.pt"
        if last_weights.exists():
            best_weights = last_weights
            print("  (using last.pt as best.pt not found)")

    print(f"\nYOLOv5 training complete.")
    print(f"  Weights: {best_weights}  (exists={best_weights.exists()})")
    return best_weights


# ===========================================================================
# STEP 4 — Train YOLOv8
# ===========================================================================
def train_yolov8():
    print("\n" + "="*60)
    print("STEP 4: Training YOLOv8n on VisDrone subset")
    print("="*60)

    yaml_path = SUBSET_DIR / "dataset.yaml"
    model = YOLO("yolov8n.pt")   # YOLOv8 nano

    results = model.train(
        data=str(yaml_path),
        epochs=EPOCHS_V8,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        project=str(ROOT / "runs"),
        name="yolov8_visdrone",
        exist_ok=True,
        verbose=True,
        workers=0,          # 0 = safer on Windows; increase on Linux/Colab
        patience=5,
        save=True,
        amp=True,           # mixed precision — faster on GPU, ignored on CPU
        cache=True,         # cache images in RAM for faster data loading
    )

    # Dynamically find where ultralytics actually saved the weights
    best_weights = ROOT / "runs" / "yolov8_visdrone" / "weights" / "best.pt"
    if not best_weights.exists():
        candidates = sorted((ROOT / "runs").rglob("best.pt"))
        v8_candidates = [p for p in candidates if "yolov8" in str(p)]
        if v8_candidates:
            best_weights = v8_candidates[-1]
        elif candidates:
            best_weights = candidates[-1]

    # Fallback to last.pt if best.pt not found
    if not best_weights.exists():
        last_weights = best_weights.parent / "last.pt"
        if last_weights.exists():
            best_weights = last_weights
            print("  (using last.pt as best.pt not found)")

    print(f"\nYOLOv8 training complete.")
    print(f"  Weights: {best_weights}  (exists={best_weights.exists()})")
    return best_weights


# ===========================================================================
# STEP 5 — Ensemble with Weighted Box Fusion (WBF)
# ===========================================================================
def ensemble_predict(yolov5_path, yolov8_path, image_path, conf=0.25, iou=0.45):
    """
    Run YOLOv5 and YOLOv8 on the same image and fuse predictions using WBF.

    WBF (Weighted Box Fusion):
      - Combines overlapping boxes from multiple models
      - Weights each model's confidence (here: equal 0.5/0.5)
      - More robust than simple NMS across models
      - Returns fused boxes, scores, and class IDs
    """
    model5 = YOLO(str(yolov5_path))
    model8 = YOLO(str(yolov8_path))

    from PIL import Image as PILImage
    with PILImage.open(str(image_path)) as pil_img:
        w, h = pil_img.size

    # Run predictions
    res5 = model5.predict(str(image_path), conf=conf, iou=iou, verbose=False)[0]
    res8 = model8.predict(str(image_path), conf=conf, iou=iou, verbose=False)[0]

    def extract(result, img_w, img_h):
        boxes, scores, labels = [], [], []
        if result.boxes is not None and len(result.boxes):
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                boxes.append([x1/img_w, y1/img_h, x2/img_w, y2/img_h])
                scores.append(float(box.conf[0]))
                labels.append(int(box.cls[0]))
        return boxes, scores, labels

    boxes5, scores5, labels5 = extract(res5, w, h)
    boxes8, scores8, labels8 = extract(res8, w, h)

    # WBF requires lists of lists (one per model)
    all_boxes   = [boxes5,  boxes8 ]
    all_scores  = [scores5, scores8]
    all_labels  = [labels5, labels8]

    if all(len(b) == 0 for b in all_boxes):
        return [], [], [], res5, res8

    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        all_boxes,
        all_scores,
        all_labels,
        weights=[0.5, 0.5],     # equal weight to both models
        iou_thr=iou,
        skip_box_thr=conf,
    )

    # Denormalize fused boxes
    fused_boxes_px = []
    for box in fused_boxes:
        x1 = int(box[0] * w)
        y1 = int(box[1] * h)
        x2 = int(box[2] * w)
        y2 = int(box[3] * h)
        fused_boxes_px.append([x1, y1, x2, y2])

    return fused_boxes_px, fused_scores.tolist(), fused_labels.tolist(), res5, res8


# ===========================================================================
# STEP 6 — Visualize Predictions
# ===========================================================================
COLORS = [
    (255,  56,  56), (255, 157,  51), (255, 112,  31), (255, 178,  29),
    (207, 210,  49), (72,  249,  10), (146, 204,  23), ( 61, 219, 134),
    ( 26, 147,  52), (  0, 212, 187),
]

def draw_boxes(img_bgr, boxes_xyxy, scores, labels, title=""):
    """Draw bounding boxes using PIL (NumPy 2.x safe)."""
    from PIL import Image as PILImage, ImageDraw, ImageFont
    import numpy as np
    # Convert BGR numpy -> RGB PIL
    img_rgb = img_bgr[:, :, ::-1].copy()
    pil = PILImage.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil)
    for box, score, label in zip(boxes_xyxy, scores, labels):
        label = int(label)
        x1, y1, x2, y2 = [int(v) for v in box]
        color = tuple(COLORS[label % len(COLORS)])
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        text = f"{CLASSES[label]} {score:.2f}"
        draw.rectangle([x1, y1 - 14, x1 + len(text) * 6, y1], fill=color)
        draw.text((x1 + 1, y1 - 13), text, fill=(255, 255, 255))
    # Return as BGR numpy for consistency
    return np.array(pil)[:, :, ::-1]


def result_to_boxes(result):
    """Extract pixel-space boxes from ultralytics Result object."""
    boxes, scores, labels = [], [], []
    if result.boxes is not None:
        for box in result.boxes:
            boxes.append([int(v) for v in box.xyxy[0].tolist()])
            scores.append(float(box.conf[0]))
            labels.append(int(box.cls[0]))
    return boxes, scores, labels


def visualize_all(image_path, yolov5_path, yolov8_path, save_path="ensemble_result.png"):
    print(f"\nVisualizing predictions on: {image_path}")

    fused_boxes, fused_scores, fused_labels, res5, res8 = ensemble_predict(
        yolov5_path, yolov8_path, image_path
    )

    from PIL import Image as PILImage
    import numpy as np
    try:
        pil_img = PILImage.open(str(image_path)).convert("RGB")
        img_bgr = np.array(pil_img)[:, :, ::-1]   # RGB -> BGR for draw_boxes
    except Exception as e:
        print(f"Could not load image: {image_path} ({e})")
        return

    # Extract individual model predictions
    boxes5, scores5, labels5 = result_to_boxes(res5)
    boxes8, scores8, labels8 = result_to_boxes(res8)

    # Draw
    img5    = draw_boxes(img_bgr, boxes5,       scores5,       labels5)
    img8    = draw_boxes(img_bgr, boxes8,       scores8,       labels8)
    img_ens = draw_boxes(img_bgr, fused_boxes,  fused_scores,  fused_labels)

    # Convert BGR->RGB for matplotlib
    imgs = [i[:, :, ::-1] for i in [img_bgr, img5, img8, img_ens]]  # BGR->RGB without cv2
    titles = ["Original", f"YOLOv5 ({len(boxes5)} dets)", f"YOLOv8 ({len(boxes8)} dets)",
              f"Ensemble/WBF ({len(fused_boxes)} dets)"]

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    fig.suptitle("VisDrone: YOLOv5 vs YOLOv8 vs Ensemble (WBF)", fontsize=14, fontweight="bold")

    for ax, im, title in zip(axes, imgs, titles):
        ax.imshow(im)
        ax.set_title(title, fontsize=11)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Visualization saved to: {save_path}")

    # Stats summary
    print("\n── Detection Summary ──────────────────────")
    print(f"  YOLOv5   : {len(boxes5):3d} detections")
    print(f"  YOLOv8   : {len(boxes8):3d} detections")
    print(f"  Ensemble : {len(fused_boxes):3d} detections  ← WBF fused result")
    if fused_scores:
        print(f"  Avg confidence (ensemble): {np.mean(fused_scores):.3f}")
    print("────────────────────────────────────────────")


# ===========================================================================
# STEP 7 — Evaluate Models
# ===========================================================================
def evaluate_models(yolov5_path, yolov8_path):
    print("\n" + "="*60)
    print("STEP 7: Model Evaluation on Validation Set")
    print("="*60)

    yaml_path = SUBSET_DIR / "dataset.yaml"

    print("\n→ YOLOv5 Validation:")
    model5 = YOLO(str(yolov5_path))
    metrics5 = model5.val(data=str(yaml_path), verbose=True)

    print("\n→ YOLOv8 Validation:")
    model8 = YOLO(str(yolov8_path))
    metrics8 = model8.val(data=str(yaml_path), verbose=True)

    # Print comparison
    print("\n" + "="*60)
    print("MODEL COMPARISON SUMMARY")
    print("="*60)
    try:
        print(f"{'Metric':<25} {'YOLOv5':>12} {'YOLOv8':>12}")
        print("-"*50)
        print(f"{'mAP@0.5':<25} {metrics5.box.map50:>12.4f} {metrics8.box.map50:>12.4f}")
        print(f"{'mAP@0.5:0.95':<25} {metrics5.box.map:>12.4f} {metrics8.box.map:>12.4f}")
        print(f"{'Precision':<25} {metrics5.box.mp:>12.4f} {metrics8.box.mp:>12.4f}")
        print(f"{'Recall':<25} {metrics5.box.mr:>12.4f} {metrics8.box.mr:>12.4f}")
    except Exception as e:
        print(f"(Could not extract metrics: {e})")

    return metrics5, metrics8


# ===========================================================================
# MAIN
# ===========================================================================
def main():
    print("="*60)
    print("  YOLOv5 + YOLOv8 + Ensemble on VisDrone Dataset")
    print("="*60)
    print(f"Dataset link : https://docs.ultralytics.com/datasets/detect/visdrone/")
    print(f"Max images   : {MAX_IMAGES}")
    print(f"YOLOv5 epochs: {EPOCHS_V5}")
    print(f"YOLOv8 epochs: {EPOCHS_V8}")
    print(f"Device       : {DEVICE}")
    print(f"Batch size   : {BATCH_SIZE}")
    print(f"Image size   : {IMG_SIZE}x{IMG_SIZE}")

    # GPU memory info
    if torch.cuda.is_available():
        print(f"\nGPU Info:")
        print(f"  Name    : {torch.cuda.get_device_name(0)}")
        print(f"  Memory  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
        print(f"  CUDA    : {torch.version.cuda}")
        torch.cuda.empty_cache()

    # Step 1: Download
    download_visdrone()

    # Step 2: Prepare subset
    prepare_subset()

    # Step 3 & 4: Train both models
    yolov5_weights = train_yolov5()
    yolov8_weights = train_yolov8()

    # Step 5 & 6: Ensemble + Visualization
    print("\n" + "="*60)
    print(f"YOLOv5 weights exists : {yolov5_weights.exists()} → {yolov5_weights}")
    print(f"YOLOv8 weights exists : {yolov8_weights.exists()} → {yolov8_weights}")
    print("="*60)

    if not yolov5_weights.exists() or not yolov8_weights.exists():
        print("\n[!] Could not find trained weights. Listing all .pt files found:")
        for pt in sorted(ROOT.rglob("*.pt")):
            print(f"    {pt}")
        print("\nPlease update yolov5_weights / yolov8_weights paths manually above.")
    else:
        # Pick a sample validation image
        val_imgs = list((SUBSET_DIR / "images" / "val").glob("*.jpg"))
        if not val_imgs:
            val_imgs = list((SUBSET_DIR / "images" / "val").glob("*.png"))

        if val_imgs:
            sample_img = random.choice(val_imgs)
            out_path   = ROOT / "ensemble_prediction.png"
            visualize_all(sample_img, yolov5_weights, yolov8_weights, save_path=str(out_path))
            print(f"\nEnsemble visualization saved: {out_path}")
        else:
            print("No validation images found for visualization.")

        # Step 7: Evaluate
        evaluate_models(yolov5_weights, yolov8_weights)

    print("\n" + "="*60)
    print("ALL DONE ✓")
    print("="*60)
    print(f"YOLOv5 weights : {yolov5_weights}")
    print(f"YOLOv8 weights : {yolov8_weights}")
    print(f"Ensemble plot  : {ROOT}/ensemble_prediction.png")
    print(f"Run logs       : {ROOT}/runs/")


if __name__ == "__main__":
    main()

⚠ No GPU found — running on CPU (will be slow)
  Tip: Use Google Colab for free GPU access
Device: cpu
PyTorch: 2.10.0+cpu
  YOLOv5 + YOLOv8 + Ensemble on VisDrone Dataset
Dataset link : https://docs.ultralytics.com/datasets/detect/visdrone/
Max images   : 300
YOLOv5 epochs: 10
YOLOv8 epochs: 10
Device       : cpu
Batch size   : 4
Image size   : 416x416

STEP 1: Downloading VisDrone Dataset
Already exists: VisDrone2019-DET-train.zip
Already exists: VisDrone2019-DET-val.zip
Already exists: VisDrone2019-DET-test-dev.zip

STEP 2: Preparing 500-image subset
Selected 300 / 6471 training images (capped at 300)
  train: 240 images
  val: 60 images
Dataset YAML written: visdrone_project/VisDrone_300/dataset.yaml

STEP 3: Training YOLOv5n on VisDrone subset
Ultralytics 8.4.17 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=No